# Boundary Qwen3-0.6B LoRA experiment
Use a GPU runtime. This notebook measures the baseline before training and does not invent results. Upload or clone the `boundary-ml` folder, then set `PROJECT_DIR` below.

In [ ]:
PROJECT_DIR = '/content/boundary-ml'  # change if needed
%cd $PROJECT_DIR
!nvidia-smi
!python --version


In [ ]:
!pip install -q -r requirements-qlora.txt
!pip install -q -e . --no-deps


In [ ]:
!python scripts/generate_dataset.py --output-dir data --variants-per-family 32


Review `data/review.csv` before treating labels as accepted. The next cell measures the unmodified model on held-out English sets.

In [ ]:
!python scripts/evaluate.py --model Qwen/Qwen3-0.6B --inputs data/test.jsonl data/challenge.jsonl --output reports/baseline.json --load-in-4bit


Run a short pipeline smoke test before the longer experiment. The smoke adapter is not an accuracy result.

In [ ]:
!python scripts/train.py --model Qwen/Qwen3-0.6B --train-file data/smoke_train.jsonl --val-file data/smoke_validation.jsonl --output-dir outputs/smoke --max-steps 8 --eval-steps 4 --load-in-4bit


In [ ]:
!python scripts/train.py --model Qwen/Qwen3-0.6B --train-file data/train.jsonl --val-file data/validation.jsonl --output-dir outputs/boundary-qwen3-0.6b-lora --epochs 3 --eval-steps 25 --load-in-4bit


In [ ]:
!python scripts/evaluate.py --model Qwen/Qwen3-0.6B --adapter outputs/boundary-qwen3-0.6b-lora --inputs data/test.jsonl data/challenge.jsonl --output reports/finetuned.json --load-in-4bit
!python scripts/compare_reports.py reports/baseline.json reports/finetuned.json --output reports/comparison.md
print(open('reports/comparison.md').read())


Run the multilingual exploratory set separately after fluent-speaker review; do not combine it with the English accuracy claim.

In [ ]:
!python scripts/evaluate.py --model Qwen/Qwen3-0.6B --adapter outputs/boundary-qwen3-0.6b-lora --inputs data/multilingual_exploratory.jsonl --output reports/multilingual_exploratory.json --load-in-4bit
